# Dependency Parsing with U-DepPLLaMA

This notebook uses the fine-tuned LLaMA-2 model from [U-DepPLLaMA](https://www.ai-lc.it/wp-content/uploads/2024/08/IJCOL_10_1_2_hromei_et_al.pdf) to perform Universal Dependency parsing as a sequence-to-sequence generation task.

The pipeline is:

1. Feed a raw sentence to the LLM through a fixed prompt template (`### Input: ... ### Answer:`).
2. The model generates a bracketed tree string such as `[ROOT[NSUBJ[He ]] [AUX [was ]] ...]`.
3. The bracketed string is decoded back into CoNLL-U style rows (`id`, `form`, `head`, `deprel`) using the helpers already shipped in [`sentence_utils.py`](sentence_utils.py).
4. The resulting dependency tree is rendered as a table and as a graph.

**Requirements**
- A CUDA GPU with enough VRAM to host a 4-bit quantized 13B model (~10GB).
- A Hugging Face account with access to the gated `meta-llama/Llama-2-13b-hf` weights, and `huggingface-cli login` run beforehand (or a token supplied below).
- The packages in [`requirements.txt`](requirements.txt) installed (see next cell).

## 1. Install dependencies

Skip this cell if you already created the `udeppllama` conda environment from the README.

In [1]:
# %pip install -r requirements.txt

In [2]:
from google.colab import files
uploaded = files.upload()

ModuleNotFoundError: No module named 'google.colab'

## 2. Imports

We reuse the repository's own `sentence_utils.py` (bracket <-> CoNLL-U conversion) and `deppllama_utils.py` (prompt template) instead of re-implementing them.

In [3]:
import sys
import os

REPO_ROOT = os.path.abspath(".")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from peft import PeftModel

# sentence_utils.py parses its own CLI args at import time (argparse), so we
# strip notebook/kernel argv noise before importing it.
_orig_argv = sys.argv
sys.argv = [_orig_argv[0]]
import sentence_utils
sys.argv = _orig_argv

import deppllama_utils

/home/paulhjwu/u-deppllama/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {PosixPath('unix')}
The following directories listed in your path were found to be non-existent: {PosixPath('VSCODE_WSL_EXT_LOCATION/up')}
The following directories listed in your path were found to be non-existent: {PosixPath('"en","resolvedLanguage"'), PosixPath('{}}'), PosixPath('"en","osLocale"'), PosixPath('"/home/paulhjwu/.vscode-server/bin/7debcd0e2acdea1c52de81bf9ee1620444407dda/out/nls.messages.json","locale"'), PosixPath('"en","availableLanguages"'), PosixPath('{"userLocale"'), PosixPath('"en","defaultMessagesFile"')}
The following directories listed in your path were found to be non-existent: {PosixPath('vs/workbench/api/node/extensionHostProcess')}
The following directories listed in your path were found to be non-existent: {PosixPath('//matplotlib_inline.backend_inline'), PosixPath('module')}
CUDA_SETUP: WARNIN

/home/paulhjwu/u-deppllama/.venv/lib/python3.10/site-packages/bitsandbytes/cuda_setup/main.py:166: UserWarning: Welcome to bitsandbytes. For bug reports, please run

python -m bitsandbytes


  warn(msg)


RuntimeError: 
        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues

## 3. (Optional) Hugging Face authentication

`meta-llama/Llama-2-13b-hf` is a gated model. If you have not already logged in via `huggingface-cli login`, uncomment and run the cell below.

In [ ]:
# from huggingface_hub import notebook_login
notebook_login()

## 4. Load the base model, the LoRA adapter, and the tokenizer

This mirrors the snippet from the [README](README.md): the LLaMA-2-13B base model is loaded in 4-bit precision (via `bitsandbytes`) and the `U-DepPLLaMA` LoRA adapter is applied on top with `peft`.

In [ ]:
BASE_MODEL = "meta-llama/Llama-2-13b-hf"
ADAPTER = "sag-uniroma2/u-depp-llama-2-13b"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

generation_config = GenerationConfig(
    num_beams=4,
    do_sample=False,
    early_stopping=True,
)

## 5. Generate the bracketed parse for a sentence

The model is prompted with `deppllama_utils.generate_prompt_pred`, which is the exact same template used at training time.

In [ ]:
def generate_parse_string(sentence, max_new_tokens=1024):
    prompt = deppllama_utils.generate_prompt_pred(sentence)
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids = inputs["input_ids"].to(model.device)

    with torch.no_grad():
        gen_outputs = model.generate(
            input_ids=input_ids,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=True,
            max_new_tokens=max_new_tokens,
            use_cache=True,
        )
    decoded = tokenizer.decode(gen_outputs.sequences[0], skip_special_tokens=True)
    return decoded.split("### Answer:")[1].strip()

In [ ]:
sentence = "He was most widely recognized for some of his books."
bracket_output = generate_parse_string(sentence)
print(bracket_output)

## 6. Decode the bracketed string into a dependency tree

`sentence_utils.get_decode` turns the bracket string back into `{id, form, head, deprel}` rows. The released checkpoint was trained on the `grct` (Grammatical-Relation-Centered Tree) representation, which is what produces the `[DEPREL[form ]]` nesting shown in the README (e.g. `[ROOT[NSUBJ[He ]] ...]`). Use `"lct"` or `"loct"` instead if you are decoding output from a model trained with a different representation.

In [ ]:
REPRESENTATION = "grct"

def decode_to_rows(bracket_string, representation_type=REPRESENTATION):
    rows = sentence_utils.get_decode(bracket_string, representation_type)
    return [rows[i] for i in sorted(rows.keys())]

def rows_to_dataframe(rows):
    return pd.DataFrame(
        [{"id": r["id"], "form": r["form"], "head": int(r["toid"]), "deprel": r["deprel"]} for r in rows]
    )

def to_conllu(rows):
    return "\n".join(sentence_utils.print_line(r) for r in rows)

rows = decode_to_rows(bracket_output)
df = rows_to_dataframe(rows)
df

In [ ]:
print(to_conllu(rows))

## 7. Visualize the dependency tree

In [ ]:
import matplotlib.font_manager as fm

_CJK_CANDIDATES = [
    "Noto Sans CJK SC", "Noto Sans CJK JP", "Noto Sans CJK TC", "Noto Sans CJK KR",
    "WenQuanYi Zen Hei", "Droid Sans Fallback", "Microsoft YaHei", "SimHei",
]
_known_fonts = {f.name for f in fm.fontManager.ttflist}
_cjk_font = next((name for name in _CJK_CANDIDATES if name in _known_fonts), None)

if _cjk_font is None:
    # matplotlib's default font (DejaVu Sans) has no CJK glyphs, so labels render
    # as empty boxes. Try installing Noto's CJK fonts; on a non-root machine this
    # silently does nothing if sudo needs a password, so install manually if needed
    # (e.g. `sudo apt-get install -y fonts-noto-cjk`) and re-run this cell.
    import subprocess
    subprocess.run(["sudo", "-n", "apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], capture_output=True)
    for path in fm.findSystemFonts():
        if "NotoSansCJK" in path or "NotoSerifCJK" in path:
            fm.fontManager.addfont(path)
    _known_fonts = {f.name for f in fm.fontManager.ttflist}
    _cjk_font = next((name for name in _CJK_CANDIDATES if name in _known_fonts), None)

if _cjk_font:
    plt.rcParams["font.sans-serif"] = [_cjk_font, "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    print(f"Using '{_cjk_font}' for CJK-capable plot labels.")
else:
    print("No CJK-capable font found/installed; Chinese/Japanese/Korean labels may render as boxes.")

In [ ]:
def hierarchy_pos(G, root, width=1.0, vert_gap=0.2, vert_loc=0, xcenter=0.5):
    """Simple tree layout: assigns x by in-order leaf position, y by depth."""

    def _next_x(counter=[0]):
        counter[0] += 1
        return counter[0]

    leaves_x = {}

    def assign_x(node, visited):
        visited.add(node)
        children = [c for c in G.successors(node) if c not in visited]
        if not children:
            leaves_x[node] = _next_x()
            return leaves_x[node]
        xs = [assign_x(c, visited) for c in children]
        leaves_x[node] = sum(xs) / len(xs)
        return leaves_x[node]

    assign_x(root, set())

    pos = {}

    def assign_y(node, depth, visited):
        visited.add(node)
        pos[node] = (leaves_x[node], -depth * vert_gap + vert_loc)
        for c in G.successors(node):
            if c not in visited:
                assign_y(c, depth + 1, visited)

    assign_y(root, 0, set())
    return pos


def visualize_tree(df, sentence=None, figsize=(11, 6)):
    G = nx.DiGraph()
    G.add_node(0, label="ROOT")
    for _, row in df.iterrows():
        G.add_node(row["id"], label=row["form"])
    for _, row in df.iterrows():
        G.add_edge(row["head"], row["id"], label=row["deprel"])

    try:
        pos = hierarchy_pos(G, root=0)
    except Exception:
        pos = nx.spring_layout(G, seed=23)

    labels = nx.get_node_attributes(G, "label")
    edge_labels = nx.get_edge_attributes(G, "label")

    plt.figure(figsize=figsize)
    nx.draw(
        G, pos, labels=labels, with_labels=True,
        node_color="#eef2f7", edgecolors="#4c72b0", node_size=1800,
        font_size=9, arrows=True, arrowsize=15,
    )
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="#b44", font_size=8)
    if sentence:
        plt.title(sentence)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


visualize_tree(df, sentence=sentence)

## 8. End-to-end helper for batches of sentences

In [ ]:
def parse_sentence(sentence, representation_type=REPRESENTATION, plot=True):
    bracket_string = generate_parse_string(sentence)
    rows = decode_to_rows(bracket_string, representation_type)
    df = rows_to_dataframe(rows)
    if plot:
        visualize_tree(df, sentence=sentence)
    return bracket_string, df


sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "She decided to study linguistics because she loves languages.",
]

results = {}
for s in sentences:
    bracket_string, df_s = parse_sentence(s)
    results[s] = df_s

## Notes

- The `U-DepPLLaMA` checkpoint used here was fine-tuned across **26 languages** from Universal Dependencies, so `parse_sentence` can be called on non-English input as well.
- Other checkpoints (different sizes / representations) may be available under the [sag-uniroma2](https://huggingface.co/sag-uniroma2) Hugging Face organization; swap `ADAPTER` and `REPRESENTATION` accordingly.
- If the model output cannot be fully decoded (e.g. truncated generation), `sentence_utils.get_decode` may raise or return a partial tree — increase `max_new_tokens` or inspect the raw `bracket_output` string first.